# Exploratory Data Analysis (EDA) — Tips Dataset

**Six plots**: two univariate, two bivariate, and two multivariate. Each plot has two numerical insights.

If using **Google Colab**, upload `tips.csv` to the session Files panel, or let `seaborn.load_dataset("tips")` download the example dataset.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 115, "savefig.dpi": 180, "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False})

file_path = Path("tips.csv")
if file_path.exists():
    df = pd.read_csv(file_path)
else:
    # In Google Colab, seaborn downloads its bundled example dataset if no local CSV was uploaded.
    df = sns.load_dataset("tips")

df["day"] = pd.Categorical(df["day"], categories=["Thur", "Fri", "Sat", "Sun"], ordered=True)
assert set(["total_bill", "tip", "sex", "smoker", "day", "time", "size"]).issubset(df.columns)
print("Data shape:", df.shape)
print("Missing values by column:\n", df.isna().sum())
print("Numerical summary:\n", df.select_dtypes(include="number").describe().round(2))


## Dataset validation
244 restaurant receipts, 7 variables; the provided CSV has no missing values. Descriptive comparisons do not establish causality.

## Chart 1: 01 Univariate Bill Histogram

In [ ]:
# 1) Univariate: distribution of total bill
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.histplot(data=df, x="total_bill", bins=18, color="#317E96", edgecolor="white", alpha=.88, ax=ax)
ax.axvline(df["total_bill"].mean(), color="#B5433A", lw=2.4, ls="--", label=f"Mean = {df['total_bill'].mean():.2f}")
ax.axvline(df["total_bill"].median(), color="#D99C3D", lw=2.4, ls=":", label=f"Median = {df['total_bill'].median():.2f}")
ax.set(title="Univariate analysis | Bill distribution", xlabel="Total bill", ylabel="Number of bills")
ax.legend(frameon=True)
fig.tight_layout()

plt.show()

**Insights:**

1. Mean bill is 19.79, above the median 17.80; the distribution has a longer right tail.
2. 188 of 244 bills (77.0%) are below 25.

## Chart 2: 02 Univariate Day Counts

In [ ]:
# 2) Univariate: frequency of observations by day
fig, ax = plt.subplots(figsize=(9, 5.2))
counts = df["day"].value_counts().reindex(["Thur", "Fri", "Sat", "Sun"])
bars = ax.bar(counts.index.astype(str), counts.values, color=["#6AA5AE", "#A7C5C5", "#2F758A", "#D59B54"], width=.7)
for bar, n in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, n+1, f"{n} ({n/len(df):.1%})", ha="center", fontsize=10, fontweight="bold")
ax.set_ylim(0, max(counts.values)*1.19)
ax.set(title="Univariate analysis | Bills by day", xlabel="Day", ylabel="Number of bills")
fig.tight_layout()

plt.show()

**Insights:**

1. Saturday has the most records (87), while Friday has the fewest (19).
2. Saturday and Sunday together account for 163 of 244 records (66.8%).

## Chart 3: 03 Bivariate Bill Tip Scatter

In [ ]:
# 3) Bivariate: total bill vs. tip
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.regplot(data=df, x="total_bill", y="tip", ax=ax,
            scatter_kws={"alpha": .55, "s": 45, "color": "#317E96"},
            line_kws={"color": "#B5433A", "lw": 2.4}, ci=None)
r = df["total_bill"].corr(df["tip"])
ax.text(.03,.94,f"Pearson r = {r:.3f}", transform=ax.transAxes,
        bbox={"boxstyle":"round,pad=.4", "facecolor":"white", "edgecolor":"#CDDBDE"})
ax.set(title="Bivariate analysis | Total bill vs. tip", xlabel="Total bill", ylabel="Tip")
fig.tight_layout()

plt.show()

**Insights:**

1. Total bill and tip show a positive Pearson correlation (r = 0.676).
2. A similar bill amount can correspond to different tip values; the relationship is not exact.

## Chart 4: 04 Bivariate Tip Boxplot Day

In [ ]:
# 4) Bivariate: tip distribution by day
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.boxplot(data=df, x="day", y="tip", order=["Thur", "Fri", "Sat", "Sun"],
            color="#80B7C1", width=.53, showfliers=True, ax=ax)
ax.set(title="Bivariate analysis | Tip distribution by day", xlabel="Day", ylabel="Tip")
fig.tight_layout()

plt.show()

**Insights:**

1. Sunday has the highest median tip (3.15); Thursday's median is 2.31.
2. The largest individual tip (10.00) occurs on Saturday.

## Chart 5: 05 Multivariate Scatter Time Party

In [ ]:
# 5) Multivariate: 4 variables (bill, tip, meal time, party size)
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.scatterplot(data=df, x="total_bill", y="tip", hue="time", size="size", sizes=(35, 220),
                palette={"Lunch":"#3B96A7", "Dinner":"#D28B51"}, alpha=.73, edgecolor="white", linewidth=.45, ax=ax)
ax.set(title="Multivariate analysis | Bill, tip, time and party size", xlabel="Total bill", ylabel="Tip")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0, title="Time / size")
fig.tight_layout()

plt.show()

**Insights:**

1. Dinner has a higher average total bill (20.80) than Lunch (17.17).
2. Bills average 28.61 for tables of four, vs. 16.45 for tables of two.

## Chart 6: 06 Multivariate Mean Tip Day Sex

In [ ]:
# 6) Multivariate: average tip by day and sex
fig, ax = plt.subplots(figsize=(9, 5.2))
sns.barplot(data=df, x="day", y="tip", hue="sex", order=["Thur", "Fri", "Sat", "Sun"],
            hue_order=["Female", "Male"], errorbar=None, palette={"Female":"#D19464", "Male":"#377E93"}, ax=ax)
means = df.groupby(["day", "sex"], observed=True)["tip"].mean()
for bar in ax.patches:
    height = bar.get_height()
    if np.isfinite(height) and height > 0.01:
        ax.text(bar.get_x()+bar.get_width()/2, height+.045, f"{height:.2f}", ha="center", fontsize=9)
ax.set_ylim(0, 4.1)
ax.set(title="Multivariate analysis | Mean tip by day and sex", xlabel="Day", ylabel="Average tip")
ax.legend(title="Sex")
fig.tight_layout()

plt.show()

**Insights:**

1. On Sunday, mean tip is 3.37 for Female vs. 3.22 for Male records.
2. On Thursday, mean tip is 2.98 for Male vs. 2.58 for Female records; sample sizes differ by group.

## Conclusion and limitations
A larger total bill tends to accompany a larger tip, while distributions differ across days and groups. The dataset is a limited sample; these are **descriptive associations**, not causal claims.